In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install transformers wandb -q

# Model 3

In [3]:
# Model 3 (DeBERTa)
# Paste this in place of your current DeBERTa training cell.
#
# Fixes vs the original notebook:
#   1. No hardcoded WandB key -- loads from Kaggle Secrets instead
#      (rotate your old key in WandB settings, it was exposed in
#      the notebook source).
#   2. Stratified train/val split -- matches Model 1 & Model 2's
#      split strategy so the three WandB runs are comparable.
#   3. Logs train/val F1 (macro) and val MAP@3, not just accuracy.
#   4. Saves the BEST checkpoint by val F1, not just the last epoch.
#   5. Adds linear warmup + a NaN-loss guard. Flat LR=2e-5 with no
#      warmup is what caused the NaN-loss collapse (Val F1 ~0.06)
#      you may have hit before -- DeBERTa-v3's disentangled attention
#      is more sensitive to large early gradients than BERT's.
#      LR is also dropped to 1e-5 as a safer starting point.

import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import wandb

# ---- WandB auth (no hardcoded keys) ----
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception as e:
    print(f"Could not load WANDB_API_KEY from Kaggle Secrets ({e}). "
          f"Falling back to offline wandb logging so the rest of the notebook still runs.")
    os.environ["WANDB_MODE"] = "offline"

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

MODEL_NAME = 'microsoft/deberta-v3-small'
MAX_LEN = 128
BATCH_SIZE = 8
EPOCHS = 3
LR = 1e-5              # lower than BERT -- DeBERTa-v3 is less stable early on
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0

wandb.init(project="24f3002284-t22026", name="model3-deberta", config={
    "model": MODEL_NAME, "max_len": MAX_LEN,
    "batch_size": BATCH_SIZE, "epochs": EPOCHS, "lr": LR
})

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
options = ['A', 'B', 'C', 'D', 'E']


def map_at_3(y_true_idx, logits):
    top3 = torch.topk(logits, 3, dim=1).indices
    scores = []
    for true_idx, pred_idx in zip(y_true_idx, top3):
        pred_list = pred_idx.tolist()
        scores.append(1.0 / (pred_list.index(true_idx) + 1) if true_idx in pred_list else 0.0)
    return sum(scores) / len(scores)


class MCQDataset(Dataset):
    def __init__(self, df, is_test=False):
        self.df = df.reset_index(drop=True)
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        encodings = []
        for opt in options:
            enc = tokenizer(str(row['prompt']), str(row[opt]),
                             max_length=MAX_LEN, padding='max_length',
                             truncation=True, return_tensors='pt')
            encodings.append({
                'input_ids': enc['input_ids'].squeeze(),
                'attention_mask': enc['attention_mask'].squeeze()
            })
        if not self.is_test:
            return encodings, torch.tensor(label_map[row['answer']], dtype=torch.long)
        return encodings


class DeBERTaMCQModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.deberta = AutoModel.from_pretrained(MODEL_NAME)
        self.classifier = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(self.deberta.config.hidden_size, 1)
        )

    def forward(self, encodings):
        logits = []
        for enc in encodings:
            output = self.deberta(
                input_ids=enc['input_ids'],
                attention_mask=enc['attention_mask']
            )
            cls = output.last_hidden_state[:, 0, :].float()  # force float32
            logits.append(self.classifier(cls))
        return torch.cat(logits, dim=1)


def collate_fn(batch):
    if isinstance(batch[0], tuple):
        encodings_list, labels = zip(*batch)
        labels = torch.stack(labels)
    else:
        encodings_list = batch
        labels = None
    batch_encodings = []
    for i in range(len(options)):
        input_ids = torch.stack([e[i]['input_ids'] for e in encodings_list])
        attention_mask = torch.stack([e[i]['attention_mask'] for e in encodings_list])
        batch_encodings.append({'input_ids': input_ids, 'attention_mask': attention_mask})
    return batch_encodings, labels


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Stratified split -- matches Model 1 & Model 2's split strategy.
train_df, val_df = train_test_split(
    train, test_size=0.2, random_state=42, stratify=train['answer']
)
train_loader = DataLoader(MCQDataset(train_df), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(MCQDataset(val_df), batch_size=BATCH_SIZE, collate_fn=collate_fn)

model = DeBERTaMCQModel().to(device).float()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

best_val_f1 = -1
nan_skips = 0
for epoch in range(EPOCHS):
    model.train()
    total_loss, n_batches = 0.0, 0
    train_true, train_pred = [], []
    for encodings, labels in train_loader:
        encodings = [{k: v.to(device) for k, v in e.items()} for e in encodings]
        labels = labels.to(device)
        optimizer.zero_grad()
        logits = model(encodings)
        loss = criterion(logits, labels)

        if not torch.isfinite(loss):
            # Skip this batch instead of letting a NaN/Inf gradient
            # corrupt every weight in the model for the rest of training.
            nan_skips += 1
            optimizer.zero_grad()
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        n_batches += 1
        train_true += labels.cpu().tolist()
        train_pred += logits.argmax(1).cpu().tolist()

    if n_batches == 0:
        raise RuntimeError(
            f"Every batch in epoch {epoch + 1} produced a non-finite loss. "
            f"Lower LR further (try 5e-6) and re-run."
        )
    if nan_skips:
        print(f"WARNING: skipped {nan_skips} non-finite-loss batches so far this run.")

    train_acc = accuracy_score(train_true, train_pred)
    train_f1 = f1_score(train_true, train_pred, average='macro')

    model.eval()
    val_true, val_pred, val_logits_all = [], [], []
    with torch.no_grad():
        for encodings, labels in val_loader:
            encodings = [{k: v.to(device) for k, v in e.items()} for e in encodings]
            labels = labels.to(device)
            logits = model(encodings)
            val_true += labels.cpu().tolist()
            val_pred += logits.argmax(1).cpu().tolist()
            val_logits_all.append(logits.cpu())

    val_acc = accuracy_score(val_true, val_pred)
    val_f1 = f1_score(val_true, val_pred, average='macro')
    val_map3 = map_at_3(val_true, torch.cat(val_logits_all, dim=0))

    print(f"Epoch {epoch+1}: Loss={total_loss/n_batches:.4f}, "
          f"Train Acc={train_acc:.4f}, Train F1={train_f1:.4f}, "
          f"Val Acc={val_acc:.4f}, Val F1={val_f1:.4f}, Val MAP@3={val_map3:.4f}")

    wandb.log({
        "epoch": epoch + 1, "loss": total_loss / n_batches, "nan_skips": nan_skips,
        "train_acc": train_acc, "train_f1": train_f1,
        "val_acc": val_acc, "val_f1": val_f1, "val_map3": val_map3,
    })

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'model3_deberta_best.pth')
        wandb.save('model3_deberta_best.pth')

wandb.summary["best_val_f1"] = best_val_f1
wandb.finish()
print("Training done!")

# ============================================================
# Inference -- loads the BEST checkpoint (by val F1).
# ============================================================
model.load_state_dict(torch.load('model3_deberta_best.pth', map_location=device))
model.eval()
test_dataset = MCQDataset(test, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn)

all_preds = []
with torch.no_grad():
    for encodings, _ in test_loader:
        encodings = [{k: v.to(device) for k, v in e.items()} for e in encodings]
        logits = model(encodings)
        top3 = torch.topk(logits, 3, dim=1).indices.cpu().numpy()
        all_preds.extend(top3)

predictions = [' '.join([options[i] for i in pred]) for pred in all_preds]
submission = pd.DataFrame({'ID': test['id'], 'Prediction': predictions})
submission.to_csv('submission.csv', index=False)
print(submission.head())
print("Done!")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: 24f3002284 (24f3002284-dl-genai-project) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run m58bvqly
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260818_030620-m58bvqly
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run model3-deberta
wandb: ⭐️ View project at https://wandb.ai/24f3002284-dl-genai-project/24f3002284-t22026
wandb: 🚀 View run at https://wandb.ai/24f3002284-dl-genai-project/24f3002284-t22026/runs/m58bvqly


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Using device: cuda


...

Epoch 1: Loss=1.6274, Train Acc=0.2406, Train F1=0.2396, Val Acc=0.5875, Val F1=0.5837, Val MAP@3=0.7192


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Epoch 2: Loss=1.2254, Train Acc=0.4919, Train F1=0.4874, Val Acc=0.8225, Val F1=0.8217, Val MAP@3=0.8962
Epoch 3: Loss=0.9120, Train Acc=0.6412, Train F1=0.6380, Val Acc=0.8800, Val F1=0.8773, Val MAP@3=0.9321


wandb: updating run metadata
wandb: uploading wandb-summary.json; uploading config.yaml
wandb: uploading config.yaml
wandb: uploading model3_deberta_best.pth; uploading output.log
wandb: uploading model3_deberta_best.pth
wandb: uploading model3_deberta_best.pth; uploading history steps 2-2, summary, console lines 22-22
wandb: uploading model3_deberta_best.pth
wandb: uploading data
wandb: 
wandb: Run history:
wandb:     epoch ▁▅█
wandb:      loss █▄▁
wandb: nan_skips ▁▁▁
wandb: train_acc ▁▅█
wandb:  train_f1 ▁▅█
wandb:   val_acc ▁▇█
wandb:    val_f1 ▁▇█
wandb:  val_map3 ▁▇█
wandb: 
wandb: Run summary:
wandb: best_val_f1 0.87732
wandb:       epoch 3
wandb:        loss 0.91199
wandb:   nan_skips 0
wandb:   train_acc 0.64125
wandb:    train_f1 0.63804
wandb:     val_acc 0.88
wandb:      val_f1 0.87732
wandb:    val_map3 0.93208
wandb: 
wandb: 🚀 View run model3-deberta at: https://wandb.ai/24f3002284-dl-genai-project/24f3002284-t22026/runs/m58bvqly
wandb: ⭐️ View project at: https://wandb.a

Training done!
   ID Prediction
0   1      A C D
1   2      B D C
2   3      B E C
3   4      A E C
4   5      C A E
Done!


In [4]:
# Inference with DeBERTa
model.eval()
test_dataset = MCQDataset(test, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn)

all_preds = []
with torch.no_grad():
    for encodings, _ in test_loader:
        encodings = [{k: v.to(device) for k, v in e.items()} for e in encodings]
        logits = model(encodings)
        top3 = torch.topk(logits, 3, dim=1).indices.cpu().numpy()
        all_preds.extend(top3)

options = ['A', 'B', 'C', 'D', 'E']
predictions = [' '.join([options[i] for i in pred]) for pred in all_preds]
submission = pd.DataFrame({'ID': test['id'], 'Prediction': predictions})
submission.to_csv('submission.csv', index=False)
print(submission.head())
print("Done!")

   ID Prediction
0   1      A C D
1   2      B D C
2   3      B E C
3   4      A E C
4   5      C A E
Done!
